In [ ]:
%matplotlib inline

# ninth_attempt.ipynb -- Diabetic Retinopathy Detection (CUSTOM category only)

**Changes vs eighth_attempt.ipynb (L-series — custom only):**
- **(L-1) Revert to plain CustomNetV2**: Remove SE blocks and residual connections. Both regressed vs plain V2 (eighth: 0.756 vs fifth: 0.770). Root cause: added complexity overfits on 2000-image dataset.
- **(L-2) 3-seed ensemble**: Train the same plain CustomNetV2 with seeds 42, 123, 456. Uniform arithmetic mean of test scores (no val-based weight search — grid search overfits the 500-val set).
- **(L-3) TVRandomVerticalFlip in train pipeline**: TTA-4 already applies vflip at inference; adding it during training aligns the training distribution with the TTA distribution.
- **(L-4) Custom-only notebook**: DenseNet121 section removed entirely — fine-tuning handled separately by cris.

**CUSTOM validity**: output_custom.csv = uniform mean of 3 plain CustomNetV2 seeds — all trained from scratch, no pretrained weights.

## 1. Imports & Setup

In [ ]:
from __future__ import print_function, division
import os, csv
import torch
import pandas as pd
from skimage import io, transform, util, color
from sklearn import metrics
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, utils
import torchvision.transforms.functional as TF
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
import time
import copy
from PIL import Image
from zipfile import ZipFile
import random
import numpy.random as npr
import cv2
import warnings

warnings.filterwarnings('ignore')
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True   # K-5 fix: NOT cudnn.enabled=False
torch.backends.cudnn.benchmark = False

plt.ion()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
DATA_ROOT = '/kaggle/input/datasets/mariamuozperez/lab5-cv'

In [ ]:
# Run once to extract data, then comment out
# import zipfile
# with zipfile.ZipFile('./db.zip', 'r') as z:
#     z.extractall('./data')

## 2. Dataset

In [ ]:
class RetinopathyDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.dataset = pd.read_csv(csv_file, header=0,
                                   dtype={'id': str, 'eye': int, 'label': int})
        if maxSize > 0:
            idx = np.random.RandomState(seed=42).permutation(range(len(self.dataset)))
            self.dataset = self.dataset.iloc[idx[:maxSize]].reset_index(drop=True)
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform
        self.levels  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
        self.classes = ['No DR', 'DR']

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = os.path.join(self.img_dir, self.dataset.id[idx] + '.jpg')
        image = io.imread(img_name)
        if self.dataset.eye[idx] == 1:
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   self.dataset.eye[idx],
            'label': (self.dataset.label[idx] > 0).astype(dtype=np.int64)
        }
        if self.transform:
            sample = self.transform(sample)
        return sample

## 3. Transforms

In [ ]:
class CropByEye(object):
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        imgray = color.rgb2gray(image)
        _, mask = cv2.threshold(imgray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = np.maximum(sidx[1].min() - self.border[1], 0)
        maxx = np.minimum(sidx[1].max() + 1 + self.border[1], w)
        miny = np.maximum(sidx[0].min() - self.border[0], 0)
        maxy = np.minimum(sidx[0].max() + 1 + self.border[1], h)
        image = image[miny:maxy, minx:maxx, ...]
        return {'image': image, 'eye': eye, 'label': label}


class BenGraham(object):
    def __init__(self, sigmaX=10):
        self.sigmaX = sigmaX

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        if image.dtype == np.uint8:
            img_u8 = image
        else:
            img_u8 = (np.clip(image, 0, 1) * 255).astype(np.uint8)
        blurred  = cv2.GaussianBlur(img_u8, (0, 0), self.sigmaX)
        enhanced = cv2.addWeighted(img_u8, 4, blurred, -4, 128)
        enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
        mask = np.zeros(enhanced.shape, dtype=np.uint8)
        h, w = enhanced.shape[:2]
        cv2.circle(mask, (w // 2, h // 2), int(0.9 * min(h, w) / 2), (1, 1, 1), -1, 8, 0)
        enhanced = enhanced * mask + 128 * (1 - mask)
        return {'image': enhanced.astype(np.float32) / 255.0, 'eye': eye, 'label': label}


class Rescale(object):
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h = self.output_size * h / w if h > w else self.output_size
            new_w = self.output_size if h > w else self.output_size * w / h
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class RandomCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = np.random.randint(0, h - new_h) if h > new_h else 0
        left = np.random.randint(0, w - new_w) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class CenterCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = int((h - new_h) / 2) if h > new_h else 0
        left = int((w - new_w) / 2) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class ToTensor(object):
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)))
        label = torch.tensor(label, dtype=torch.long)
        return {'image': image, 'eye': eye, 'label': label}


class Normalize(object):
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        dtype = image.dtype
        mean = torch.as_tensor(self.mean, dtype=dtype, device=image.device)
        std  = torch.as_tensor(self.std,  dtype=dtype, device=image.device)
        image.sub_(mean[:, None, None]).div_(std[:, None, None])
        return {'image': image, 'eye': eye, 'label': label}


class TVCenterCrop(object):
    def __init__(self, size):
        self.CC = transforms.CenterCrop(size)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.CC(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomHorizontalFlip(object):
    def __init__(self, p=0.5):
        self.flip = transforms.RandomHorizontalFlip(p=p)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.flip(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomVerticalFlip(object):
    """(L-3) Align training distribution with the vflip pass in 4-pass TTA."""
    def __init__(self, p=0.5):
        self.flip = transforms.RandomVerticalFlip(p=p)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.flip(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomRotation(object):
    def __init__(self, degrees=15):
        self.rotate = transforms.RandomRotation(degrees=degrees)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.rotate(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVColorJitter(object):
    def __init__(self, brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05):
        self.jitter = transforms.ColorJitter(
            brightness=brightness, contrast=contrast,
            saturation=saturation, hue=hue)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.jitter(pil)))
        return {'image': image, 'eye': eye, 'label': label}

## 4. Data Pipelines & DataLoaders

In [ ]:
pixel_mean = [0.485, 0.456, 0.406]
pixel_std  = [0.229, 0.224, 0.225]

# (L-3) TVRandomVerticalFlip added after hflip — aligns training dist with TTA-4 vflip pass.
train_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(256),
    TVRandomHorizontalFlip(p=0.5),
    TVRandomVerticalFlip(p=0.5),
    TVRandomRotation(degrees=15),
    TVColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    RandomCrop(224),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

eval_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(256),
    CenterCrop(224),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

train_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'train.csv'),
    root_dir=DATA_ROOT, maxSize=0, transform=train_transform)
val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'val.csv'),
    root_dir=DATA_ROOT, transform=eval_transform)
test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'test.csv'),
    root_dir=DATA_ROOT, transform=eval_transform)
print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}')

In [ ]:
_sample = train_dataset[0]
_img = _sample['image'].numpy().transpose(1, 2, 0)
print(f'Pipeline output -- dtype: {_img.dtype}, min: {_img.min():.3f}, max: {_img.max():.3f}, mean: {_img.mean():.3f}')
assert abs(_img.mean()) < 0.3, f'BenGraham all-gray bug! mean={_img.mean():.3f}'
print('Sanity check passed.')

In [ ]:
train_labels_bin_for_sampler = (train_dataset.dataset['label'].values > 0).astype(int)
class_counts   = np.bincount(train_labels_bin_for_sampler)
sample_weights = np.where(train_labels_bin_for_sampler == 1,
                          1.0 / class_counts[1], 1.0 / class_counts[0])
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_dataset), replacement=True)

train_dataloader = DataLoader(train_dataset, batch_size=64,  sampler=sampler,  num_workers=0)
val_dataloader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=0)
test_dataloader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

train_labels_bin = (train_dataset.dataset['label'].values > 0).astype(int)
n_neg = int((train_labels_bin == 0).sum())
n_pos = int((train_labels_bin == 1).sum())
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
print(f'No-DR: {n_neg}  DR: {n_pos}  pos_weight: {pos_weight.item():.3f}')

criterion      = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
image_datasets = {'train': train_dataset, 'val': val_dataset}
dataloaders    = {'train': train_dataloader, 'val': val_dataloader}
dataset_sizes  = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names    = train_dataset.classes

## 5. Training & Evaluation Utilities

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=7, label_smoothing=0.0):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_auc, best_epoch, no_improve = 0.0, -1, 0

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            numSamples = dataset_sizes[phase]
            outputs_m  = np.zeros((numSamples,), dtype=float)
            labels_m   = np.zeros((numSamples,), dtype=int)
            running_loss, contSamples = 0.0, 0
            for sample in dataloaders[phase]:
                inputs    = sample['image'].to(device).float()
                labels    = sample['label'].to(device).float()
                batchSize = labels.shape[0]
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    logits = model(inputs).flatten()
                    if label_smoothing > 0.0 and phase == 'train':
                        labels_ls = labels * (1 - label_smoothing) + label_smoothing / 2.0
                        loss = criterion(logits, labels_ls)
                    else:
                        loss = criterion(logits, labels)
                    scores = torch.sigmoid(logits).detach()
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * batchSize
                outputs_m[contSamples:contSamples + batchSize] = scores.cpu().numpy()
                labels_m [contSamples:contSamples + batchSize] = labels.cpu().numpy()
                contSamples += batchSize
            if phase == 'train':
                scheduler.step()
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_auc  = metrics.roc_auc_score(labels_m, outputs_m)
            print('{} Loss: {:.4f}  AUC: {:.4f}'.format(phase, epoch_loss, epoch_auc))
            if phase == 'val':
                if epoch_auc > best_auc:
                    best_auc, best_epoch, no_improve = epoch_auc, epoch, 0
                    best_model_wts = copy.deepcopy(model.state_dict())
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f'Early stopping: no improvement for {patience} epochs.')
                        model.load_state_dict(best_model_wts)
                        return model
        print()
    elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(elapsed // 60, elapsed % 60))
    print('Best model: epoch {:d}  val AUC: {:.4f}'.format(best_epoch, best_auc))
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
def eval_val_auc(model, name, tta=False):
    model.eval()
    n = len(val_dataset)
    scores_m = np.zeros((n, 1), dtype=float)
    labels_m = np.zeros((n,), dtype=int)
    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))    # hflip
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))    # vflip
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3]))) # rot180
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            scores_m[cont:cont + bs, :] = out.cpu().numpy()
            labels_m[cont:cont + bs]     = sample['label'].numpy()
            cont += bs
    auc    = metrics.roc_auc_score(labels_m, scores_m)
    suffix = ' (TTA-4)' if tta else ''
    print(f'{name}{suffix}  --  val AUC: {auc:.4f}')
    return auc


def get_val_scores(model, tta=False):
    """Return (scores_array, labels_array) for val set."""
    model.eval()
    n = len(val_dataset)
    scores_m = np.zeros((n, 1), dtype=float)
    labels_m = np.zeros((n,), dtype=int)
    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            scores_m[cont:cont + bs, :] = out.cpu().numpy()
            labels_m[cont:cont + bs]     = sample['label'].numpy()
            cont += bs
    return scores_m, labels_m


def test_model(model, tta=False):
    model.eval()
    n = len(test_dataset)
    outputs_m = np.zeros((n, 1), dtype=float)
    cont = 0
    with torch.no_grad():
        for sample in test_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            outputs_m[cont:cont + bs, :] = out.cpu().numpy()
            cont += bs
    return outputs_m

---
## 6. Plain CustomNetV2 (CUSTOM category)

**(L-1)** Exact fifth_attempt architecture — no SEBlock, no residuals. 5-block 3×3 same-padding CNN with
Global Average Pooling. Channels: 3→32→64→128→256→256. ~1.05M params.

SE blocks and residual connections both regressed vs this baseline (eighth_attempt: 0.756 vs fifth_attempt: 0.770).
Root cause: added complexity overfits on the 2000-image training set.

In [ ]:
class CustomNetV2(nn.Module):
    """Plain CustomNetV2: 5-block 3x3 CNN with Global Average Pooling. No pretrained weights."""

    def __init__(self):
        super().__init__()

        def _block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
            )

        self.features = nn.Sequential(
            _block(3,   32),   # 224 -> 112
            _block(32,  64),   # 112 ->  56
            _block(64,  128),  #  56 ->  28
            _block(128, 256),  #  28 ->  14
            _block(256, 256),  #  14 ->   7
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

In [ ]:
_net = CustomNetV2().to(device)
_inp = next(iter(train_dataloader))['image'].to(device).float()
with torch.no_grad():
    _out = _net(_inp)
print(f'CustomNetV2  Input: {_inp.shape}  Output: {_out.shape}')
total = sum(p.numel() for p in _net.parameters())
print(f'CustomNetV2 -- total params: {total:,}')
del _net, _inp, _out

---
## 7. Multi-Seed Training (seeds 42, 123, 456)

**(L-2)** Train the same plain CustomNetV2 three times with different seeds.
Diversity comes from stochastic weight init, batch ordering, and augmentation.
No architectural changes — avoids the overfitting risk that caused SE/residual regressions.

Val scores and test scores are collected during each run; models are deleted after to free GPU memory.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    npr.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
seeds           = [42, 123, 456]
val_aucs        = {}
val_scores_list = []
val_labels_ref  = None
test_scores     = {}

for seed in seeds:
    print(f'\n{"="*60}')
    print(f'Training CustomNetV2  seed={seed}')
    print(f'{"="*60}')
    set_seed(seed)

    model     = CustomNetV2().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-3)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    model = train_model(model, criterion, optimizer, scheduler,
                        num_epochs=50, patience=10, label_smoothing=0.0)

    torch.save(model.state_dict(), f'best_customnetv2_seed{seed}.pth')
    print(f'Saved: best_customnetv2_seed{seed}.pth')

    auc = eval_val_auc(model, f'CustomNetV2 seed={seed}', tta=True)
    val_aucs[seed] = auc

    vs, vl = get_val_scores(model, tta=True)
    val_scores_list.append(vs)
    if val_labels_ref is None:
        val_labels_ref = vl

    test_scores[seed] = test_model(model, tta=True)

    del model, optimizer, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

---
## 8. Ensemble & Validation Summary

Uniform arithmetic mean — no val-based weight search.
Grid search on 500 val samples would overfit (eighth_attempt demonstrated this).
For a same-architecture ensemble the equal-weight mean is asymptotically optimal.

In [ ]:
print('=== Per-seed validation AUC (TTA-4) ===')
for seed in seeds:
    print(f'  seed={seed}: {val_aucs[seed]:.4f}')

# seed=42 should be close to fifth_attempt val AUC (~0.796)
if abs(val_aucs[42] - 0.796) > 0.012:
    print(f'WARNING: seed=42 val AUC {val_aucs[42]:.4f} differs from expected ~0.796 by >0.012.')
    print('         Check that cudnn.deterministic=True (not cudnn.enabled=False).')

mean_val_scores  = np.mean(np.stack(val_scores_list, axis=0), axis=0)
auc_ensemble_val = metrics.roc_auc_score(val_labels_ref, mean_val_scores)
print(f'\n  3-seed ensemble val AUC (TTA-4): {auc_ensemble_val:.4f}')

outputs_custom = np.mean(np.stack([test_scores[s] for s in seeds], axis=0), axis=0)

assert outputs_custom.shape == (1000, 1)
assert np.isfinite(outputs_custom).all()
print(f'\nCustom score range: [{outputs_custom.min():.4f}, {outputs_custom.max():.4f}]')
print('Checks passed.')

---
## 9. Generate output_custom.csv

**CUSTOM category**: plain CustomNetV2, 3-seed uniform mean, 4-pass TTA. No pretrained weights.

In [ ]:
with open('output_custom.csv', mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_custom)
print('Written: output_custom.csv')

In [ ]:
with ZipFile('./codabench_submission.zip', 'w') as zf:
    zf.write('./output_custom.csv')
print('Created: codabench_submission.zip')

print(f'\nFinal summary (ninth_attempt.ipynb):')
for seed in seeds:
    print(f'  CustomNetV2 seed={seed}  val AUC (TTA-4): {val_aucs[seed]:.4f}')
print(f'  3-seed ensemble val AUC (TTA-4): {auc_ensemble_val:.4f}')